In [ ]:

from dotenv import load_dotenv
import os
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
# Define model
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
model = ChatGroq(groq_api_key = groq_api_key, model_name = "llama-3.3-70b-versatile")
model


In [ ]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, Myname is Sebastian and I am a Chief AI Engineer")])

In [ ]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hi, My name is Sebastian and I am a Chief AI Engineer"),
    AIMessage(content = "Nice to meet you, Sebastian. It's great to connect with a Chief AI Engineer like yourself. What brings you here today? Are you working on an exciting AI project, or do you have any questions or topics related to AI that you'd like to discuss? I'm all ears!"),
    HumanMessage(content = "Hey, what is my name and what do I do?")
    ])


In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
# Function to retrieve history for specific session id
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Interact with our model and chat history
with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [ ]:
config = {"configurable":{"session_id":"chat1"}}

response = with_message_history.invoke([
    HumanMessage(content = "Hi, My name is Sebastian and I am a Chief AI Engineer")],
    config = config
    )
response.content

In [ ]:
with_message_history.invoke([
    HumanMessage(content = "What is my name?")],
    config = config
    )

In [ ]:
## Change the context, session id
config2 = {"configurable":{"session_id":"chat2"}}
with_message_history.invoke([
    HumanMessage(content = "What is my name?")],
    config = config2
    )

## Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(

    [
        ("system","You are a helpful assitant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name = "messages")
    ]
)

chain = prompt|model

In [ ]:
chain.invoke({"messages":[HumanMessage(content = "Hi, my names is Sebastian")]})

In [ ]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [ ]:
config = {"configurable":{"session_id":"chat1"}}
response = with_message_history.invoke(
    [HumanMessage(content = "What is my name?")],config = config,
)
response.content

In [ ]:
# Add more complexity

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(

    [
        ("system","You are a helpful assitant. Answer all the question to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name = "messages")
    ]
)

chain = prompt|model

In [ ]:
response = chain.invoke(
    {"messages":[HumanMessage(content = "Hi, My name is Sebastian")],"language":"Spanish"}
)
response.content

In [ ]:
# More complex chax in a message history class
with_message_history = RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [ ]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {"messages":[HumanMessage(content = "Hi, I am Carlos")],"language":"German"},
    config = config
)
response.content

# Manage conversation history
If left unmanaged it will grow unbounded and overflow the context windows

In [ ]:
from langchain_core.messages import SystemMessage, trim_messages 
# Helper to reduce how many messages we're sending to the model
trimmer = trim_messages(max_tokens = 45, strategy = 'last', 
                        token_counter = model,
                        include_system = True, 
                        allow_partial = False,
                        start_on = "human")


messages = [
SystemMessage(content = "You are a good assistant"),
HumanMessage(content = "Hi, I'm Bob"),
AIMessage(content = "Hi!"),
HumanMessage(content = "I like vanilla Ice cream"),
AIMessage(content = "Nice"),
HumanMessage(content = "What's 2+2"),
AIMessage(content = "4"),
HumanMessage(content = "Thanks"),
AIMessage(content = "No problem"),
HumanMessage(content = "Having fun?"),
AIMessage(content = "Yes!"),
]

trimmer.invoke(messages)

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages")|trimmer)|prompt|model
)

chain.invoke({"messages":messages + [HumanMessage(content = "What Ice cream do I like?")],"language":"german"})

In [ ]:
chain.invoke({"messages":messages + [HumanMessage(content = "What math problem did I ask?")],"language":"german"})

In [ ]:
# Lets wrap it in the message History

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config = {"configurable":{"session_id":"chat5"}}


In [ ]:
response = with_message_history.invoke({
    "messages":[HumanMessage(content = "What math problem did I ask?")],"language":"italian"
}, config = config)
response.content

# Vector stores and retrievers

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content = "Dogs are great companions, know for their loyalty and friendliness.",
        metadata = {"source":"mammal-pets-doc"}
    ),
    Document(
        page_content = "Cats are independent pets that often enjoy their own space",
        metadata = {"source":"mammal-pets-doc"}
    ),
    Document(
        page_content = "Goldfish are popular pets for beginners, requiring relatively simple care",
        metadata = {"source":"fish-pets-doc"}
    ),
    Document(
        page_content = "Parrots are intelligent birds capable of mimicking human speech",
        metadata = {"source":"birds-pets-doc"}
    )
    ]



In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HUGGINGFACE_TOKEN"] = os.getenv("HUGGINGFACE_TOKEN")

llm = ChatGroq(groq_api_key = groq_api_key, model= "llama-3.1-8b-instant")
llm

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents,embedding = embeddings)


In [ ]:
vectorstore.similarity_search("cat")

In [ ]:
# Async query
await vectorstore.asimilarity_search("cat")

In [ ]:
## Vector stores are not runnable (cannot be integrated into Langchain expression language), retrievers are runnables
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])


In [ ]:
## Retriever method #2
retriever = vectorstore.as_retriever(search_type = "similarity",
                         search_kwargs = {"k":1})
retriever.batch(["cat","dog"])


In [ ]:
# RAGCreate a chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only

{question}

Context:

{context}
"""
prompt = ChatPromptTemplate.from_messages([("human",message)])
# Runnable pass through means it will get passed in the invoke method
rag_chain = {"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response = rag_chain.invoke("Tell me about dogs")
response.content